In [100]:
import pprint
import glob
import random
import os
import pandas as pd
import numpy as np
import geopandas as gpd
import rasterio
from shapely.geometry import Polygon


### Steps
- process FAOSTAT data to have values for every year 2000-2020
- load some forestloss patches
- get the time range 
- use it to query the tabular data

## Process FAOSTAT data (incl. normalization)

In [69]:
df = pd.read_csv('../data/tmp/faostat_data_per_year.csv')

In [70]:
df.Item.unique()

array(['Cattle', 'Soya beans', 'Cocoa beans', 'Coffee, green',
       'Oil palm fruit', 'Natural rubber in primary forms'], dtype=object)

In [78]:
FAOSTAT_classes = ['Cattle', 'Oil palm fruit', 'Soya beans', 'Cocoa beans', 'Coffee, green',
       'Natural rubber in primary forms']

In [79]:
country_name_mapping = {
    "Vietnam" : "Viet Nam",
    "Tanzania" : "United Republic of Tanzania",
    "Dem. Rep. Congo" : "Democratic Republic of the Congo",
    "Dominican Rep." : "Dominican Republic",
    "Bolivia" : "Bolivia (Plurinational State of)",
    "Venezuela" : "Venezuela (Bolivarian Republic of)",
    "Central African Rep.": "Central African Republic",
    "Eq. Guinea" : "Equatorial Guinea",
    "eSwatini" : "Eswatini",
    "Laos" : "Lao People's Democratic Republic",
    "Taiwan" : "China, Taiwan Province of",
    "Brunei" : "Brunei Darussalam",
    "S. Sudan" : "South Sudan",
    "Solomon Is." : "Solomon Islands"
}

In [80]:
names_to_edit = [key for key in country_name_mapping.keys()]
for name in names_to_edit:
    tropical_countries.loc[tropical_countries['name'] == name, 'name'] = country_name_mapping[name]
    

In [81]:
elements = {
    'Cattle' : {}, 
    'Oil palm fruit' : {}, 
    'Soya beans' : {}, 
    'Cocoa beans' : {}, 
    'Coffee, green' : {}, 
    'Natural rubber in primary forms' : {}
}

for idx, row in tropical_countries.iterrows():
    
    #try:
    #    country_name = country_name_mapping[row['name']]
    #except KeyError:
    country_name = row['name']
    
    _df = df.loc[df.Area == country_name]
    if len(_df) == 0:
        print('No FAOSTAT data found for ', row['name'])
    for _cls in FAOSTAT_classes:
        commodity_df = _df.loc[_df.Item == _cls]
        elements_available = commodity_df.Element.unique()
        
        for element in elements_available:
            try:
                elements[_cls][element] += 1
            except KeyError:
                elements[_cls][element] = 1

No FAOSTAT data found for  W. Sahara
No FAOSTAT data found for  Iran
No FAOSTAT data found for  Somaliland


In [84]:
df_filtered = df.loc[
    (df.Item == 'Cattle') |
    ((df.Item == 'Oil palm fruit') & (df.Element == 'Area harvested' )) |
    ((df.Item == 'Cocoa beans') & (df.Element == 'Area harvested' )) |
    ((df.Item == 'Coffee, green') & (df.Element == 'Area harvested' )) |
    ((df.Item == 'Natural rubber in primary forms') & (df.Element == 'Area harvested' )) |
    ((df.Item == 'Soya beans') & (df.Element == 'Area harvested' )) 
]

In [67]:
# 2.71 Mha - area under cultivation of cocoa for 2021 by Kalischek 

#I should probably interpolate for the earlier years as well, 
#it does not make sense to have a massive jump between 2019 and 2020

#df_filtered.loc[
#    ((df_filtered.Area == 'Ghana') & 
#     (df_filtered.Year == 2020) & 
#     (df_filtered.Item == "Cocoa beans")), 'Value'] = 2710000


In [122]:
df_filtered.loc[df_filtered.Element == 'Area harvested']

,Domain Code,Domain,Area Code (M49),Area,Element Code,Element,Item Code (CPC),Item,Year Code,Year,Unit,Value,Flag,Flag Description,Note
42,QCL,Crops and livestock products,8,Albania,5312,Area harvested,141.0,Soya beans,2000,2000,ha,400.0,A,Official figure,NaN
45,QCL,Crops and livestock products,8,Albania,5312,Area harvested,141.0,Soya beans,2001,2001,ha,400.0,A,Official figure,NaN
48,QCL,Crops and livestock products,8,Albania,5312,Area harvested,141.0,Soya beans,2002,2002,ha,200.0,A,Official figure,NaN
51,QCL,Crops and livestock products,8,Albania,5312,Area harvested,141.0,Soya beans,2003,2003,ha,413.0,A,Official figure,NaN
54,QCL,Crops and livestock products,8,Albania,5312,Area harvested,141.0,Soya beans,2004,2004,ha,300.0,A,Official figure,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24346,QCL,Crops and livestock products,716,Zimbabwe,5312,Area harvested,141.0,Soya beans,2016,2016,ha,39935.0,A,Official figure,NaN
24349,QCL,Crops and livestock products,716,Zimbabwe,5312,Area harvested,141.0,Soya beans,2017,2017,ha,23515.0,A,Official figure,NaN
24352,QCL,Crops and livestock products,716,Zimbabwe,5312,Area harvested,141.0,Soya beans,2018,2018,ha,37307.0,A,Official figure,NaN
24355,QCL,Crops and livestock products,716,Zimbabwe,5312,Area harvested,141.0,Soya beans,2019,2019,ha,33136.0,A,Official figure,NaN


In [132]:
cattle_values = df_filtered.loc[df_filtered.Element == 'Stocks'].Value
area_prod = df_filtered.loc[df_filtered.Element == 'Area harvested'].Value

In [ ]:
numer_cols = 

In [135]:
#y = (x - mean) / standard_deviation

#Normalize separately cattle and the other production classes 
for i, values in enumerate([cattle_values, area_prod]):

    nanmean = np.nanmean(values)
    nanstd = np.nanstd(values)
    
    
    #Normalize
    normed_values = (values - nanmean) / nanstd
    
    
    #Avoid having negative values (0 often symbolizes missing values; there should be no negative values)
    normed_values = normed_values - normed_values.min()
    
    if i == 0:
        df_filtered.loc[df_filtered.Element == 'Stocks', "Value"] = normed_values
    elif i == 1:
        df_filtered.loc[df_filtered.Element == 'Area harvested', "Value"] = normed_values










In [151]:
df_filtered = df_filtered.rename(columns={'Area': 'country'})

In [152]:
df_filtered.to_csv('../data/tmp/faostat_reports_2000_2020_per_year.csv', index=False)

### Compute tabular values from TMF patches

then test and compare with static vals

In [145]:
labels = pd.read_csv('/home/jan/Documents/EPFL/project_2/data/labels/dataset_v4/labels_55k.csv')

forestloss_dir = '/home/jan/Documents/EPFL/project_2/data/forestloss_tiles/dataset_v4'
all_patches = glob.glob(forestloss_dir + "/*")

#while True:
for i in range(50):
    row = labels.sample(1)
    forestloss_path = os.path.join(forestloss_dir, f'{row.sampleid.values[0]}.tif')
    with rasterio.open(forestloss_path) as src:
        data = src.read()

    assert np.all(np.mod(data, 1) == 0)
    years = data[(data >= 2000) & (data <= 2020)]
    _min, _max = years.min(), years.max()
    
    country_data = df_filtered.loc[df_filtered.country == row.country.values[0]]
    
    values = []
    for _class in FAOSTAT_classes:
        try:
            production_at_the_start = country_data.loc[(country_data.Item == _class) & (country_data.Year == int(_min)), 'Value' ].item()
            production_at_the_end = country_data.loc[(country_data.Item == _class) & (country_data.Year == int(_max)), 'Value' ].item()
            expansion = production_at_the_end - production_at_the_start
        except ValueError: #Signifies that one (or both) of the values are missing
            expansion = 0
        values.append(expansion)

In [146]:
values = []
for _class in FAOSTAT_classes:
    try:
        production_at_the_start = country_data.loc[(country_data.Item == _class) & (country_data.Year == int(_min)), 'Value' ].item()
        production_at_the_end = country_data.loc[(country_data.Item == _class) & (country_data.Year == int(_max)), 'Value' ].item()
        expansion = production_at_the_end - production_at_the_start
    except ValueError: #Signifies that one (or both) of the values are missing
        expansion = 0
    values.append(expansion)



In [147]:
values

[-0.030897478704509762,
 0.03836682994164915,
 0.005852126152788406,
 0.026028756778713413,
 0.00791548861777741,
 0.0]

In [119]:
FAOSTAT_classes

['Cattle',
 'Oil palm fruit',
 'Soya beans',
 'Cocoa beans',
 'Coffee, green',
 'Natural rubber in primary forms']

In [88]:
df_filtered.loc[df_filtered.Area.str.startswith('Vie')]

,Domain Code,Domain,Area Code (M49),Area,Element Code,Element,Item Code (CPC),Item,Year Code,Year,Unit,Value,Flag,Flag Description,Note
23767,QCL,Crops and livestock products,704,Viet Nam,5111,Stocks,2111.0,Cattle,2000,2000,An,4127872.0,A,Official figure,NaN
23768,QCL,Crops and livestock products,704,Viet Nam,5111,Stocks,2111.0,Cattle,2001,2001,An,3899683.0,A,Official figure,NaN
23769,QCL,Crops and livestock products,704,Viet Nam,5111,Stocks,2111.0,Cattle,2002,2002,An,4062966.0,A,Official figure,NaN
23770,QCL,Crops and livestock products,704,Viet Nam,5111,Stocks,2111.0,Cattle,2003,2003,An,4394400.0,A,Official figure,NaN
23771,QCL,Crops and livestock products,704,Viet Nam,5111,Stocks,2111.0,Cattle,2004,2004,An,4907710.0,A,Official figure,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23968,QCL,Crops and livestock products,704,Viet Nam,5312,Area harvested,141.0,Soya beans,2016,2016,ha,99578.0,A,Official figure,NaN
23971,QCL,Crops and livestock products,704,Viet Nam,5312,Area harvested,141.0,Soya beans,2017,2017,ha,67993.0,A,Official figure,NaN
23974,QCL,Crops and livestock products,704,Viet Nam,5312,Area harvested,141.0,Soya beans,2018,2018,ha,53364.0,A,Official figure,NaN
23977,QCL,Crops and livestock products,704,Viet Nam,5312,Area harvested,141.0,Soya beans,2019,2019,ha,49217.0,A,Official figure,NaN


In [103]:
def _get_commodity_expansion_rates(country, min_year, max_year=2020): #TODO: Max year should be 2022, this is temporary
        
    FAOSTAT_classes = ['Cattle', 'Palm oil', 'Soya beans', 'Cocoa beans', 'Coffee, green',
   'Natural rubber in primary forms']

    #TODO: this needs to be filtered so that for every commodity and year, there is only 1 element (either area, or production etc.)

    country_data = df_filtered.loc[df_filtered.Area == country]

    values = []

    for _class in FAOSTAT_classes:
        try:
            production_at_the_start = country_data.loc[(country_data.Item == _class) & (country_data.Year == int(min_year)), 'Value' ].item()
            production_at_the_end = country_data.loc[(country_data.Item == _class) & (country_data.Year == int(max_year)), 'Value' ].item()
            expansion = production_at_the_end - production_at_the_start
        except ValueError: #Signifies that one (or both) of the values are missing
            expansion = 0
        values.append(expansion)

    return values

In [104]:
vals = {}

for country in tropical_countries.name.unique():
    vals[country] = _get_commodity_expansion_rates(country, min_year=2000)

In [105]:
FAOSTAT_classes = ['Cattle', 'Palm oil', 'Soya beans', 'Cocoa beans', 'Coffee, green',
'Natural rubber in primary forms']

In [106]:
tmp_df = pd.DataFrame(vals).T
tmp_df.columns = FAOSTAT_classes

In [107]:
tmp_df

,Cattle,Palm oil,Soya beans,Cocoa beans,"Coffee, green",Natural rubber in primary forms
Fiji,-241699.000,0.000,0.000,-639.000,-8.000,0.000
United Republic of Tanzania,13551841.000,-740.000,36162.000,16484.000,116960.000,0.000
W. Sahara,0.000,0.000,0.000,0.000,0.000,0.000
United States of America,-4405700.000,0.000,4125820.000,0.000,0.000,0.000
Papua New Guinea,4668.000,344000.000,0.000,-2474.000,-43357.000,662.000
...,...,...,...,...,...,...
Somaliland,0.000,0.000,0.000,0.000,0.000,0.000
Uganda,8549833.000,0.000,-54000.000,55465.000,349885.000,0.000
Rwanda,876376.000,0.000,18945.000,0.000,-5147.000,0.000
Trinidad and Tobago,5575.000,0.000,0.000,-11278.000,765.000,0.000


In [108]:
areas = []
for idx, row in tmp_df.iterrows():
    _name = row.name
    area = tropical_countries.loc[tropical_countries['name'] == _name]['area'].item()
    areas.append(area)

tmp_df['country_area'] = areas

In [109]:
tmp_df
tmp_df.loc[tmp_df.index == "Viet Nam"]



,Cattle,Palm oil,Soya beans,Cocoa beans,"Coffee, green",Natural rubber in primary forms,country_area
Viet Nam,2102647.000,0.000,-82536.000,0.000,160663.000,497264.000,336019.374


In [110]:
def normalize_by_col(df, cols_to_normalize, norm_column):
    """For each column in df from cols_to_normalize, divide by the value in norm_column. Return normalized df"""
    df_normalized = df.copy()
    for col in cols_to_normalize:
        df_normalized[col] = df_normalized[col] / df_normalized[norm_column]
    return df_normalized

In [111]:
cols = ['Cattle', 'Palm oil', 'Soya beans', 'Cocoa beans', 'Coffee, green',
       'Natural rubber in primary forms',]

In [112]:
norm_df = normalize_by_col(tmp_df, cols, 'country_area')

In [113]:
norm_df

,Cattle,Palm oil,Soya beans,Cocoa beans,"Coffee, green",Natural rubber in primary forms,country_area
Fiji,-12.531,0.000,0.000,-0.033,-0.000,0.000,19287.604
United Republic of Tanzania,14.528,-0.001,0.039,0.018,0.125,0.000,932779.346
W. Sahara,0.000,0.000,0.000,0.000,0.000,0.000,96669.250
United States of America,-0.463,0.000,0.434,0.000,0.000,0.000,9509850.934
Papua New Guinea,0.010,0.741,0.000,-0.005,-0.093,0.001,464531.614
...,...,...,...,...,...,...,...
Somaliland,0.000,0.000,0.000,0.000,0.000,0.000,167473.218
Uganda,34.789,0.000,-0.220,0.226,1.424,0.000,245760.253
Rwanda,37.508,0.000,0.811,0.000,-0.220,0.000,23365.244
Trinidad and Tobago,0.720,0.000,0.000,-1.457,0.099,0.000,7738.923


### Calculate statistics from training set
- std and mean for cattle (unit: stocks), palm oil (unit: tons - production), rest (area harvested)

In [114]:
pd.set_option('display.float_format', lambda x: '%.3f' % x)

In [115]:
### Doing it this way results in 0 values to become non-zero:
#y = (x - mean) / standard_deviation

'''stds = norm_df.std()
means = norm_df.mean()

harvested_area = norm_df[['Soya beans', 'Cocoa beans', 'Coffee, green',
       'Natural rubber in primary forms']].values.flatten()


mean = harvested_area.mean()
std = harvested_area.std()

normalization_training_dst = {
    'Cattle' : {
        'mean' : 2.541,
        'std' : 8.711
    },
    'Palm oil' : {
        'mean': 0.649,
        'std' : 3.125
    },
    
    'harvested_area' : {
        'mean' : 0.079,
        'std': 0.636
    }
    
}

norm_df['Cattle'] = (norm_df['Cattle'] - normalization_training_dst['Cattle']['mean'])/normalization_training_dst['Cattle']['std']
norm_df['Palm oil'] = (norm_df['Palm oil'] - normalization_training_dst['Palm oil']['mean'])/normalization_training_dst['Palm oil']['std']

for _cls in ['Soya beans', 'Cocoa beans', 'Coffee, green',
       'Natural rubber in primary forms']:
    
    norm_df[_cls] = (norm_df[_cls] - normalization_training_dst['harvested_area']['mean'])/normalization_training_dst['harvested_area']['std']

'''


"stds = norm_df.std()\nmeans = norm_df.mean()\n\nharvested_area = norm_df[['Soya beans', 'Cocoa beans', 'Coffee, green',\n       'Natural rubber in primary forms']].values.flatten()\n\n\nmean = harvested_area.mean()\nstd = harvested_area.std()\n\nnormalization_training_dst = {\n    'Cattle' : {\n        'mean' : 2.541,\n        'std' : 8.711\n    },\n    'Palm oil' : {\n        'mean': 0.649,\n        'std' : 3.125\n    },\n    \n    'harvested_area' : {\n        'mean' : 0.079,\n        'std': 0.636\n    }\n    \n}\n\nnorm_df['Cattle'] = (norm_df['Cattle'] - normalization_training_dst['Cattle']['mean'])/normalization_training_dst['Cattle']['std']\nnorm_df['Palm oil'] = (norm_df['Palm oil'] - normalization_training_dst['Palm oil']['mean'])/normalization_training_dst['Palm oil']['std']\n\nfor _cls in ['Soya beans', 'Cocoa beans', 'Coffee, green',\n       'Natural rubber in primary forms']:\n    \n    norm_df[_cls] = (norm_df[_cls] - normalization_training_dst['harvested_area']['mean'])/

### Normalization v2

just divide by max value for each of palm oil, cattle, area harvested

In [116]:
max_vals = norm_df.max()

In [117]:
norm_df['Cattle'] = norm_df['Cattle']/max_vals['Cattle']
norm_df['Palm oil'] = norm_df['Palm oil']/max_vals['Palm oil']


max_area = max_vals[['Soya beans', 'Cocoa beans', 'Coffee, green',
       'Natural rubber in primary forms']].max()

for _cls in ['Soya beans', 'Cocoa beans', 'Coffee, green',
       'Natural rubber in primary forms']:
    
    norm_df[_cls] = norm_df[_cls]/max_area


Index(['Fiji', 'United Republic of Tanzania', 'W. Sahara',
       'United States of America', 'Papua New Guinea', 'Indonesia',
       'Argentina', 'Chile', 'Democratic Republic of the Congo', 'Somalia',
       ...
       'Morocco', 'Egypt', 'Libya', 'Ethiopia', 'Djibouti', 'Somaliland',
       'Uganda', 'Rwanda', 'Trinidad and Tobago', 'South Sudan'],
      dtype='object', length=113)

In [120]:
norm_df['country'] = norm_df.index

In [122]:
norm_df.to_csv('../data/tmp/faostat_reports.csv', index=False)

## Read and explore the normalized data

In [2]:
norm_df = pd.read_csv('../data/tmp/faostat_reports.csv')

In [82]:
#norm_df['geometry'] = gpd.GeoSeries.from_wkt(norm_df['geometry'])

#gdf = gpd.GeoDataFrame(norm_df)

In [3]:
tmp = norm_df.set_index('country')

In [7]:
tmp

,Cattle,Palm oil,Soya beans,Cocoa beans,"Coffee, green",Natural rubber in primary forms,country_area
country,,,,,,,
Fiji,-0.334100,0.000000,0.000000,-0.004772,-0.000060,0.000000,1.928760e+04
United Republic of Tanzania,0.387346,-0.000032,0.005584,0.002545,0.018061,0.000000,9.327793e+05
W. Sahara,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,9.666925e+04
United States of America,-0.012352,0.000000,0.062491,0.000000,0.000000,0.000000,9.509851e+06
Papua New Guinea,0.000268,0.030182,0.000000,-0.000767,-0.013444,0.000205,4.645316e+05
...,...,...,...,...,...,...,...
Somaliland,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.674732e+05
Uganda,0.927525,0.000000,-0.031649,0.032508,0.205067,0.000000,2.457603e+05
Rwanda,1.000000,0.000000,0.116790,0.000000,-0.031730,0.000000,2.336524e+04


In [4]:
tmp['Cocoa beans'].sort_values(ascending=False)[:25]

country
Côte d'Ivoire                         1.000000
Ghana                                 0.715853
Sierra Leone                          0.175377
Liberia                               0.126110
Ecuador                               0.071527
Cameroon                              0.064903
Indonesia                             0.060100
Dominican Republic                    0.059033
Nigeria                               0.035406
Solomon Islands                       0.033441
Uganda                                0.032508
Guinea                                0.025280
Peru                                  0.015003
Colombia                              0.013159
Nicaragua                             0.011978
Philippines                           0.009474
Congo                                 0.008556
Timor-Leste                           0.007128
Costa Rica                            0.007110
Panama                                0.003973
India                                 0.003750
Venez

In [24]:
norm_df.to_csv('../data/tmp/faostat_reports.csv', index=False)

In [22]:
norm_df['country'] = tropical_countries['name'].reset_index(drop=True)

In [9]:
norm_df.loc[norm_df['country'] == "Malaysia"]



,Cattle,Palm oil,Soya beans,Cocoa beans,"Coffee, green",Natural rubber in primary forms,country_area,country
97,-0.003103,1.0,0.0,-0.029833,-0.011306,-0.124159,338220.757641,Malaysia


In [103]:
df_filtered['Flag Description'].unique()

array(['Official figure', 'Estimated value',
       'Missing value (data cannot exist, not applicable)',
       'Imputed value', 'Figure from international organizations'],
      dtype=object)

In [74]:
df = pd.read_csv("/home/jan/Documents/EPFL/project_2/data/labels/dataset_v4/labels_55k.csv")
df['geometry'] = gpd.GeoSeries.from_wkt(df['geometry'])

gdf = gpd.GeoDataFrame(df)

In [78]:
names = gdf.country.unique()

In [79]:
_dict = {}
test_countries = ['Ecuador', 'Malaysia', 'Thailand']
for name in names:
    if name not in test_countries:
        _dict[name] = "train"
    else:
        _dict[name] = "val"

In [80]:
import pprint
pprint.pprint(_dict)

{'Angola': 'train',
 'Australia': 'train',
 'Bahamas': 'train',
 'Bangladesh': 'train',
 'Belize': 'train',
 'Benin': 'train',
 'Bhutan': 'train',
 'Bolivia (Plurinational State of)': 'train',
 'Brazil': 'train',
 'Brunei Darussalam': 'train',
 'Burundi': 'train',
 'Cambodia': 'train',
 'Cameroon': 'train',
 'Central African Republic': 'train',
 'China': 'train',
 'Colombia': 'train',
 'Congo': 'train',
 'Costa Rica': 'train',
 'Cuba': 'train',
 "Côte d'Ivoire": 'train',
 'Democratic Republic of the Congo': 'train',
 'Dominican Republic': 'train',
 'Ecuador': 'val',
 'El Salvador': 'train',
 'Equatorial Guinea': 'train',
 'Ethiopia': 'train',
 'Fiji': 'train',
 'France': 'train',
 'Gabon': 'train',
 'Ghana': 'train',
 'Guatemala': 'train',
 'Guinea': 'train',
 'Guyana': 'train',
 'Haiti': 'train',
 'Honduras': 'train',
 'India': 'train',
 'Indonesia': 'train',
 'Jamaica': 'train',
 'Kenya': 'train',
 "Lao People's Democratic Republic": 'train',
 'Liberia': 'train',
 'Madagascar': 'trai